# VoyageAI - Transport Agent

This notebook creates the Transport Agent for VoyageAI.

The Transport Agent is responsible for:
1. Reading the selected destination from the Destination Agent
2. Understanding the user's travel style and budget preference
3. Suggesting intercity transport options
4. Suggesting local transport options
5. Returning structured transport recommendations

This version does not use live flight/train/bus APIs.
It gives general transport guidance only.

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import json

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

e:\Agentic AI\VoyageAI Backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
load_dotenv()

print("Environment variables loaded.")

In [2]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "agents":
    PROJECT_ROOT = CURRENT_DIR.parent.parent
elif CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

CHROMA_DIR = PROJECT_ROOT / "chroma_db"

print("Current directory:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Chroma DB folder:", CHROMA_DIR)

if not CHROMA_DIR.exists():
    raise FileNotFoundError("Chroma DB folder not found. Run 01_rag_ingestion.ipynb first.")

Current directory: e:\Agentic AI\VoyageAI Backend\notebooks\agents
Project root: e:\Agentic AI\VoyageAI Backend
Chroma DB folder: e:\Agentic AI\VoyageAI Backend\chroma_db


In [4]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model loaded successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4427.09it/s]


Embedding model loaded successfully.


In [5]:
vector_store = Chroma(
    persist_directory=str(CHROMA_DIR),
    embedding_function=embedding_model,
    collection_name="voyageai_travel_knowledge"
)

print("ChromaDB loaded successfully.")
print("Total vectors:", vector_store._collection.count())

ChromaDB loaded successfully.
Total vectors: 45


In [6]:
def get_available_destinations():
    collection_data = vector_store._collection.get(include=["metadatas"])

    destinations = set()

    for metadata in collection_data["metadatas"]:
        city = metadata.get("city")
        if city:
            destinations.add(city.lower().strip())

    return sorted(list(destinations))


available_destinations = get_available_destinations()

print("Available destinations in ChromaDB:")
print(available_destinations)

Available destinations in ChromaDB:
['agra', 'andaman', 'darjeeling', 'delhi', 'goa', 'jaipur', 'kashmir', 'kerala', 'ladakh', 'manali', 'mumbai', 'pondicherry', 'rishikesh', 'udaipur', 'varanasi']


In [7]:
def normalize_text(text: str):
    if not text:
        return ""

    return text.lower().strip()


def is_destination_available(destination_name: str):
    destination_name = normalize_text(destination_name)

    if not destination_name:
        return False

    for destination in available_destinations:
        if destination == destination_name:
            return True

        if destination in destination_name:
            return True

        if destination_name in destination:
            return True

    return False

In [8]:
DESTINATION_KEY_MAP = {
    "goa": "goa",
    "jaipur": "jaipur",
    "manali": "manali",
    "kerala": "kerala",
    "andaman": "andaman",
    "andaman and nicobar islands": "andaman",
    "udaipur": "udaipur",
    "rishikesh": "rishikesh",
    "varanasi": "varanasi",
    "ladakh": "ladakh",
    "kashmir": "kashmir",
    "darjeeling": "darjeeling",
    "pondicherry": "pondicherry",
    "puducherry": "pondicherry",
    "agra": "agra",
    "delhi": "delhi",
    "mumbai": "mumbai"
}


def normalize_destination_to_key(destination_name: str):
    if not destination_name:
        return None

    destination_name = destination_name.lower().strip()

    for name, key in DESTINATION_KEY_MAP.items():
        if name in destination_name:
            return key

    return destination_name.replace(" ", "_")

In [9]:
def get_transport_context(user_query: str, destination_result: dict, k: int = 4):
    destination = destination_result.get("recommended_destination")
    city_key = normalize_destination_to_key(destination)

    retrieval_query = f"""
    Destination: {destination}
    User request: {user_query}
    Need transport suggestions, local travel tips, travel convenience, trip duration, destination access, and movement advice.
    """

    try:
        docs = vector_store.similarity_search(
            retrieval_query,
            k=k,
            filter={"city": city_key}
        )

        if not docs:
            docs = vector_store.similarity_search(retrieval_query, k=k)

    except Exception as e:
        print("Filter search failed. Running normal similarity search.")
        print("Error:", e)
        docs = vector_store.similarity_search(retrieval_query, k=k)

    context_parts = []

    for i, doc in enumerate(docs, start=1):
        city = doc.metadata.get("city")
        source = doc.metadata.get("source")

        context = f"""
Context {i}
City: {city}
Source: {source}
Content:
{doc.page_content}
"""
        context_parts.append(context.strip())

    return "\n\n".join(context_parts)

In [10]:
user_query = "I want to travel from Bhubaneswar to Goa on a low budget"

destination_result = {
    "agent_name": "Destination Agent",
    "status": "success",
    "recommended_destination": "Goa",
    "reason": "Goa matches beaches, seafood, nightlife and relaxed coastal travel.",
    "suitable_for": ["beach lovers", "food lovers", "nightlife travelers"],
    "suggested_duration": "3 to 5 days",
    "best_time_to_visit": "November to February",
    "confidence": "high"
}

transport_context = get_transport_context(user_query, destination_result)

print(transport_context)

Context 1
City: goa
Source: goa.txt
Content:
Destination: Goa
State/Region: Goa
Destination Type: Beach, nightlife, seafood, coastal vacation, friends trip, couples trip

Overview:
Goa is one of India's most popular beach destinations. It is known for beaches, nightlife, seafood, Portuguese heritage, forts, churches, markets, and relaxed coastal culture.

Best For:
- Beach lovers
- Seafood lovers
- Friends groups
- Couples
- Nightlife travelers
- Relaxed vacation seekers

Context 2
City: goa
Source: goa.txt
Content:
Best Time to Visit:
November to February is the most comfortable and popular season. Monsoon is scenic but beach activities may be limited.

Ideal Duration:
3 to 5 days

Travel Tips:
- Rent a scooter for local travel if comfortable.
- North Goa is better for nightlife.
- South Goa is better for peaceful beaches and luxury stays.

Context 3
City: goa
Source: goa.txt
Content:
Top Attractions:
- Baga Beach
- Calangute Beach
- Anjuna Beach
- Vagator Beach
- Fort Aguada
- Chapor

In [11]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2
)

print("Groq LLM loaded successfully.")

Groq LLM loaded successfully.


In [12]:
transport_agent_prompt = ChatPromptTemplate.from_template("""
You are the Transport Agent of VoyageAI.

Your task is to suggest transport options for the selected destination based on:
1. User travel request
2. Destination Agent output
3. Retrieved destination knowledge

User Travel Request:
{user_query}

Destination Agent Output:
{destination_result}

Retrieved Travel Knowledge:
{context}

Return your answer strictly in valid JSON format.

JSON structure:
{{
  "agent_name": "Transport Agent",
  "status": "success",
  "source_city": "source city if mentioned, otherwise unknown",
  "destination": "destination name",
  "travel_style_detected": "budget/comfort/luxury/fastest/unknown",
  "recommended_intercity_mode": "flight/train/bus/car/ferry/mixed/unknown",
  "intercity_options": [
    {{
      "mode": "flight/train/bus/car/ferry/mixed",
      "suitability": "high/medium/low",
      "best_for": "who this mode is best for",
      "cost_level": "low/medium/high/unknown",
      "time_level": "slow/medium/fast/unknown",
      "note": "short practical note"
    }}
  ],
  "local_transport_options": ["option 1", "option 2", "option 3"],
  "transport_tips": ["tip 1", "tip 2", "tip 3"],
  "limitations": "mention that this version does not use live schedules or prices",
  "confidence": "high/medium/low"
}}

Rules:
- Use the selected destination from the Destination Agent.
- Use retrieved travel knowledge when destination-specific transport tips are available.
- You may give general transport guidance, but do not invent exact train names, flight numbers, bus operators, departure times, or live prices.
- If source city is not clearly mentioned, set source_city to "unknown".
- If exact transport data is missing, be transparent in the limitations field.
- Do not add markdown.
- Do not wrap JSON in ```json.
""")

In [13]:
transport_agent_chain = transport_agent_prompt | llm | StrOutputParser()

print("Transport Agent chain created successfully.")

Transport Agent chain created successfully.


In [14]:
def parse_json_response(response: str):
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        try:
            start = response.find("{")
            end = response.rfind("}") + 1
            json_str = response[start:end]
            return json.loads(json_str)
        except Exception:
            print("Failed to parse JSON. Raw response:")
            print(response)
            return {
                "error": "Failed to parse JSON",
                "raw_response": response
            }

In [15]:
def run_transport_agent(user_query: str, destination_result: dict, k: int = 4):
    destination_status = destination_result.get("status")
    destination = destination_result.get("recommended_destination")

    if destination_status != "success":
        return {
            "agent_name": "Transport Agent",
            "status": "skipped",
            "destination": None,
            "message": "Transport Agent skipped because Destination Agent did not return a valid destination.",
            "reason": destination_result.get("reason"),
            "confidence": "low"
        }

    if not destination:
        return {
            "agent_name": "Transport Agent",
            "status": "no_destination_found",
            "destination": None,
            "message": "Transport Agent cannot run because no destination was provided.",
            "confidence": "low"
        }

    if not is_destination_available(destination):
        return {
            "agent_name": "Transport Agent",
            "status": "out_of_knowledge_base",
            "destination": destination,
            "message": f"{destination} is not available in the current VoyageAI knowledge base.",
            "available_destinations": available_destinations,
            "confidence": "low"
        }

    context = get_transport_context(
        user_query=user_query,
        destination_result=destination_result,
        k=k
    )

    response = transport_agent_chain.invoke({
        "user_query": user_query,
        "destination_result": json.dumps(destination_result, indent=2),
        "context": context
    })

    parsed_response = parse_json_response(response)

    parsed_response["status"] = parsed_response.get("status", "success")

    return parsed_response

In [17]:
user_query = "I want a honeymoon trip to Andaman with clean beaches, island hopping and water activities"

destination_result = {
    "agent_name": "Destination Agent",
    "status": "success",
    "recommended_destination": "Andaman and Nicobar Islands",
    "reason": "Andaman is suitable for clean beaches, scuba diving, snorkeling and island experiences.",
    "suitable_for": ["honeymoon couples", "beach lovers", "water sports lovers"],
    "suggested_duration": "5 to 7 days",
    "best_time_to_visit": "October to May",
    "confidence": "high"
}

transport_result = run_transport_agent(user_query, destination_result)

print(json.dumps(transport_result, indent=2))

{
  "agent_name": "Transport Agent",
  "status": "success",
  "source_city": "unknown",
  "destination": "Andaman and Nicobar Islands",
  "travel_style_detected": "luxury",
  "recommended_intercity_mode": "ferry",
  "intercity_options": [
    {
      "mode": "ferry",
      "suitability": "high",
      "best_for": "beach lovers, honeymoon couples, scuba diving beginners, snorkeling lovers, nature lovers, luxury island travelers",
      "cost_level": "medium",
      "time_level": "medium",
      "note": "Ferry transfers can be booked in advance to ensure availability."
    },
    {
      "mode": "flight",
      "suitability": "medium",
      "best_for": "those in a hurry, luxury travelers",
      "cost_level": "high",
      "time_level": "fast",
      "note": "Flights are available from major Indian cities to Port Blair, the capital of Andaman and Nicobar Islands."
    },
    {
      "mode": "mixed",
      "suitability": "low",
      "best_for": "adventurous travelers",
      "cost_level

In [18]:
user_query = "I want to go to Venice"

destination_result = {
    "agent_name": "Destination Agent",
    "status": "out_of_knowledge_base",
    "recommended_destination": None,
    "reason": "Venice is not available in the current VoyageAI knowledge base.",
    "confidence": "low"
}

transport_result = run_transport_agent(user_query, destination_result)

print(json.dumps(transport_result, indent=2))

{
  "agent_name": "Transport Agent",
  "status": "skipped",
  "destination": null,
  "message": "Transport Agent skipped because Destination Agent did not return a valid destination.",
  "reason": "Venice is not available in the current VoyageAI knowledge base.",
  "confidence": "low"
}
